<a href="https://colab.research.google.com/github/vera2005/MLCV_course1/blob/main/Lesson2/ML_Course_HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Задача 1**
**Ручной градиентный спуск для линейной регрессии на California Housing**

In [12]:
# Базовые библиотеки
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Загрузка датасета California Housing
from datasets import load_dataset

# Работа с выборками и масштабирование
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Метрики качества регрессии
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Готовая линейная регрессия через стохастический градиентный спуск
from sklearn.linear_model import SGDRegressor

# Фиксация генератора случайных чисел для воспроизводимости
np.random.seed(42)

**Что дано**


Датасет California Housing — реальные данные о районах Калифорнии. Он вам уже знаком

 В нем 16 640 объектов (районов).

 У каждого 8 признаков: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude.

1 целевая переменная: MedHouseVal — медианная стоимость дома (в единицах 100 000 $).



**Что требуется**

Обучить линейную регрессию вручную, без использования готовых моделей (LinearRegression, SGDRegressor, Dense из Keras).

Модель: ŷ = w₁x₁ + w₂x₂ + ... + w₈x₈ + b

в матричном виде: ŷ = X @ w + b



Т.е. **требуется**  найти такие w (8 чисел) и b (1 число), при которых MSE минимальна:

MSE = (1/N) * Σ (yᵢ - ŷᵢ)²

Это показывает, что происходит внутри model.fit() и SGDRegressor.fit() — тот же самый градиентный спуск, только написанный вручную.

Инженерно расчленяем библиотечные моедли и избавляемся от магии вне хогвартса

Далее часть работы будет дублировать занятие в классе. А именно: загрузка данных, разбивка на выборки, предобработка(StandardScaler ). Это были шаги 1 - 4. Предлагаю Вам реализовать их самостоятельно с опрой на блокнот занятия в классе.

1. Загрузка даных

2. Выведите первые 5 образцов из датасета


3. Разбивка на выборки

4. Масштабирование признаков
После масштабирования выведите еще раз первые 5 образцов и убедитесь, что признаки масштабируются.

5. Инициализация w, b
Требуется: инифиализировать веса w и b случайными значениями. Отметим, что число весов соответсвует числу признаков в датасете.

 (В нашем случае n_samples, n_features = X_train.shape. Где  n_samples = 13312 - число образцов в обучающей выборке, n_features = 8 - число признаков у каждого образца)

Инициализация шага обучения, числа эпох и аналог history - список, куда мы будем складывать значения функций потерь в каждой эпохи. Число эпох и шаг обучения Вы можете задать самостоятельно)

In [ ]:
lr = 0.01
epochs = 100
loss_history = []

6. Цикл обучения.
Требуется: написать цикл for, который имитирует проход по эпохам. В рамках цикла необходимо:

- посчитать предсказанное значение для всех образцов с текущими весами
- посчтать значение функции потерь (MSE, MAE)
- записать полученное значение функции потерь loss в наш список loss_history (loss_history.append(loss))
-посчитать чатсные производные по каждому из параметров (весов и смещения)

Подсказка:     dw = -2 / n_samples * X_train.T @ (y_train - y_pred) - для w как массив из 8 чисел

- скорректировать значение весов и смещения (w -= lr * dw)


7. Визуализация

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(loss_history, color='blue', linewidth=2)
plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.title("Ручной GD: кривая обучения")
plt.yscale('log')      # логарифмическая шкала — лучше видно падение
plt.grid(True, alpha=0.3)
plt.show()

8. Оценка на тесте.
Требуется:
-высчитать предсказанное значение для образцов из тестовой выборки
- Посчтать ошибку MSE/MAE  

(можно использовать from sklearn.metrics import mean_squared_error, mean_absolute_error)

print("MSE:", mean_squared_error(y_test, y_pred_test))

print("MAE:", mean_absolute_error(y_test, y_pred_test))






9.Попытка предсказать цену дома для района из тестовой выборки

Требуется:
- посчитать предсказанное значение с весами, полученными в ходе обучения модели


In [ ]:
y_pred_test = X_test @ w + b

# Сравним первые 10 предсказаний с реальными ценами
print("Первые 10 предсказаний vs реальных значений:")
comparison = pd.DataFrame({
    'Реальная цена': y_test[:10].round(3),
    'Предсказание':  y_pred_test[:10].round(3),
    'Ошибка':        (y_pred_test[:10] - y_test[:10]).round(3)
})
print(comparison)

#**Задача 2**
**Бинарная классификация на датасете IMDB (отзывы о фильмах)**

**Цель**
Освоить бинарную классификацию с функцией потерь binary_crossentropy на примере текстовых отзывов к фильмам.

**Постановка**
Датасет IMDB: 50 000 отзывов, метка 1 — положительный, 0 — отрицательный.
Нужно построить нейросеть, которая по тексту отзыва предсказывает его тональность. Это классический пример бинарной классификации — в отличие от регрессии, где мы предсказывали число (цену дома), здесь мы предсказываем вероятность принадлежности к классу.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Воспроизводимость
np.random.seed(42)
tf.random.set_seed(42)

# Настройки графиков
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

In [2]:
!pip install -q datasets

1. Загрузка датасета

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

Этот вывод показывает структуру датасета IMDB, который вы загрузили. DatasetDict — это словарь, в котором лежат несколько готовых подвыборок. Разберём его по частям.

train — обучающая выборка (25 000)
Это та часть, на которой учится модель. Мы будем использовать её для model.fit

test — тестовая выборка (25 000)
Это отложенные данные, которые модель не видит при обучении. Используется только в конце — для честной оценки качества

unsupervised — неразмеченная выборка (50 000)
Это дополнительный набор из 50 000 отзывов без меток (только text, метки формальные). Он предназначен для полуобучения (semi-supervised learning) или для предобучения языковых моделей.

В нашей задаче он не нужен.

2. Разбивка датасета на задачи и ответы. И разбивка на тестовую и обучающую выборки.

train_texts - задачки для обучения
train_labels - ответы для обучения
test_texts, test_labels  - для тестирования

In [4]:
train_texts  = dataset['train']['text']
train_labels =
test_texts   =
test_labels  = dataset['test']['label']

Превратите train_labels и test_labels в массивы numpy

In [ ]:
y_train =
y_test  =

print(f"Обучающих отзывов: {len(train_texts)}")
print(f"Тестовых отзывов:   {len(test_texts)}")
print(f"Форма y_train: {y_train.shape}, y_test: {y_test.shape}")
print(f"Распределение меток в train (0, 1): {np.bincount(y_train)}")

**3. Токенизация**

Токенизация — это превращение текста в числа. Нейросеть не умеет работать со словами, ей нужны цифры. Токенизатор строит словарь: каждому слову присваивает свой номер.

"the movie is good" → [1, 42, 7, 315]

сеть может умножать числа, считать градиенты, обновлять веса. С текстом она этого делать не может. Поэтому мы заменяем слова на их индексы в словаре.

In [7]:
vocab_size = 10000
max_len    = 200

**vocab_size** = 10000 — максимальный размер словаря. Мы оставляем только 10 000 самых частотных слов. Редкие слова отбрасываются. Это уменьшает размер модели и борется с шумом.

**max_len** = 200 — максимальная длина отзыва в словах. Всё, что длиннее — обрежется; всё, что короче — дополнится нулями.

In [8]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

print(f"Всего уникальных слов в train: {len(tokenizer.word_index)}")
print(f"Оставляем только {vocab_size} самых частых")

Всего уникальных слов в train: 88583
Оставляем только 10000 самых частых


Создаём объект **Tokenizer**:

num_words=10000 — токенизатор будет вести словарь только на 10 000 слов. Если слово встречается реже — оно игнорируется.

oov_token="<OOV>" — Out Of Vocabulary: специальный токен для слов, которых нет в словаре. Всё, что не вошло в топ-10 000, будет заменено на <OOV>. Это важно, потому что тестовые данные могут содержать слова, которых не было в train.

**fit_on_texts** - только строит словарь. Он не превращает тексты в числа. Это будет позже, в texts_to_sequences

**Обучение** токенизатора на train-текстах:

1) Токенизатор проходит по всем 25 000 отзывов.

2) Разбивает каждый текст на слова.

3) Считает частоту каждого слова.

4) Сортирует слова по убыванию частоты.

6) Присваивает самые частые слова — маленькие номера (1, 2, 3, ...).


**4. Паддинг**  - два ключевых шага: превращает слова в индексы и приводит все отзывы к одинаковой длине. Только после этого данные можно подать в нейросеть.

Токенизатор уже построил словарь {слово: индекс} на предыдущем шаге. Теперь texts_to_sequences проходит по каждому тексту и заменяет слова на их индексы.

По примеру для train_sequences напишите для test_sequences

In [9]:
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences  =

Важно: длины у всех последовательностей разные. Один отзыв может быть 30 слов, другой — 500. Нейросеть такое «рваное» представление не принимает. Поэтому нужен следующий шаг

Если слово не вошло в топ-10 000, токенизатор заменит его на индекс oov_token="<OOV>". Если в тексте встретилось слово, которого вообще не было при fit, тоже подставится <OOV>.

**pad_sequences** - Функция приводит все последовательности к длине max_len = 200:

- если последовательность короче 200 — добавляет нули (padding);

- если длиннее 200 — обрезает (truncating).


Аналогично по примеру X_train напишите X_test

In [10]:
X_train = pad_sequences(train_sequences, maxlen=max_len,
                        padding='post', truncating='post')
X_test  =

padding='post'	'post'	Нули добавляются в конец (после слов)

truncating='post'	'post'	Лишние слова отбрасываются с конца

In [ ]:
print(f"Форма X_train: {X_train.shape}")
print(f"Форма X_test:  {X_test.shape}")

25000 - число отзывов
200 - длина отзыва в словах, т.е. 200 чисел, кодирующих слова из отзыва по сларю токенизатора

**6. СОЗДАНИЕ МОДЕЛИ**

Через контейнер Sequential создайте модель для классификации тональности отзывов

Какие слои нам потребуются?

**1) Embedding** — «смысл» слов в числах
**Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len)**

На вход модели подаются индексы слов: [12, 87, 5, 42, ...]. Но индекс — это просто номер, он не несёт смысла. Слово «movie» имеет индекс 42, слово «film» — индекс 318. С точки зрения математики между этими числами нет никакой связи, хотя по смыслу слова близки.

Embedding решает эту проблему: он превращает каждый индекс в вектор из 16 вещественных чисел. И эти векторы учатся во время тренировки — похожие по смыслу слова получают похожие векторы.

Слой дает Таблицу размером 10 000 × 16 — векторное представление всего словаря. Это самые «тяжёлые» параметры в модели: 160 000 чисел, которые учатся.


**2) GlobalAveragePooling1D** — усредняем по словам

После Embedding у нас получилась матрица формы (batch, 200, 16): 200 векторов по 16 чисел — по одному вектору на каждое слово. Но классификатор хочет один вектор на весь отзыв, а не 200.
GlobalAveragePooling1D делает простое: усредняет все 200 векторов по словам. Результат — вектор формы (batch, 16)

Полученный вектор из 16 чисел призван отразить средний смысл отзыва

**3, 4) 2 полносвязных слоя**. Один для вычислений, один выходной. В выходном будет 1 нейрон и какая функция активация используется для задачи бинарной классификации?)


In [16]:
model =

**7. Компиляция**

Используем метод compile.
В качестве оптимизатора можно использовать Adam,

функция потерь для бинарной классификации?)


В качестве метрики можно использовать точность - Она показывает долю правильных ответов среди всех объектов.
 = (правильные ответы/все ответы)

metrics=['accuracy'] в этот аргумент передается список метрик, в нашем случае в списке будет одна


**8) ОБУЧЕНИЕ**

 Создайте переменную history (правило хорошего тона), которой будет присвоен результат метода fit.

 В качестве аргументов необходимо передать: обучающую выборку и ответы к ней, число эпох, размер батча, размер валидационной выборки (например 0.2)

*Код ниже имеет основной целью общее ознакомительное значение, в нем не нужно глубоко разбираться) Но если возникает инетерс - можете покапаться*

*Если возникают вопросы - буду рада ответить, по любой чатси кода*

**9. Визуализация**

Левый график - функция потерь
2 кривые: синяя - ошибка на обучающей выборке, по которой обновлялись веса
оранжевая - ошибка на валидации

Если кривые падают, то модель учится.

Когда кривые выходят на плато - обучение сошлось, дальше улучшения нет, это оптимум.

Когда val loss начинает расти, а train loss продолжает падать → переобучение: модель запоминает train вместо поиска закономерностей.

Правый график - метрика - точность

Синяя - доля правильных ответов на обучающей выборке
Ораневая - на валидации

Кривые растут → модель улучшается.

Если train accuracy намного выше val accuracy → переобучение.

Если обе близки и обе высокие → модель обобщает (это то, что нам нужно).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='train loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='val loss',   linewidth=2)
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Binary Crossentropy")
axes[0].set_title("Функция потерь (train vs val)")
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='train accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='val accuracy',   linewidth=2)
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Точность (train vs val)")
axes[1].legend()

plt.tight_layout()
plt.show()

**10. Оценка на тестовой выборке**

Проверка обученной модели на тестовой выборке.

evaluate - метод обученной моедли, который
1) прогоняет через моедль все объекты из X_test.
2) Сравнивает предсказания с реальными y_test.
3) Считает функцию потерь и все метрики, указанные при compile.


Важно: evaluate не обучает модель. Веса не меняются. Это просто «прогон» и подсчёт метрик.


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")

**11) Предсказание для 5 образцов**

predict использует обученную модель для предсказания ответов.

In [ ]:
n_show   = 5
X_sample = X_test[:n_show]
y_true   = y_test[:n_show]

y_proba  = model.predict(X_sample, verbose=0).flatten()
#verbose не показывает прогресс-бар, т.е. отладочную информацию по процессу

y_pred   = (y_proba > 0.5).astype(int)
abs_err  = np.abs(y_true - y_proba)

result = pd.DataFrame({
    'Текст (первые 70 символов)': [t[:70] + "..." for t in test_texts[:n_show]],
    'Реальная метка':            y_true,
    'Вероятность (sigmoid)':     y_proba.round(4),
    'Предсказание':              y_pred,
    'Ошибка |y - ŷ|':            abs_err.round(4),
    'Правильно?':                (y_pred == y_true)
})
print(result.to_string(index=True))

# Средняя ошибка по 5 образцам
print(f"\nСредняя ошибка |y - ŷ| по {n_show} образцам: {abs_err.mean():.4f}")


12) Метрики на всей тестовой выборке

In [ ]:
y_pred_proba_all = model.predict(X_test, verbose=0).flatten()
y_pred_all       = (y_pred_proba_all > 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred_all)
prec = precision_score(y_test, y_pred_all)
rec  = recall_score(y_test, y_pred_all)
f1   = f1_score(y_test, y_pred_all)

print(f"Accuracy  (точность):       {acc:.4f}  ({acc * 100:.2f}%)")
print(f"Precision (точность+):      {prec:.4f}")
print(f"Recall    (полнота):        {rec:.4f}")
print(f"F1-score  (гармоническое):  {f1:.4f}")

13) БОНУС - проверка модели на своих примерах)

In [ ]:


def predict_sentiment(text, model, tokenizer, max_len=200):
    seq    = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len,
                           padding='post', truncating='post')
    proba  = model.predict(padded, verbose=0)[0][0]
    label  = "ПОЛОЖИТЕЛЬНЫЙ" if proba > 0.5 else "ОТРИЦАТЕЛЬНЫЙ"
    return label, proba

my_reviews = [
    # Напишите тут свои предложения на английском. Каждое предложение в
]

for review in my_reviews:
    label, proba = predict_sentiment(review, model, tokenizer)
    print(f"Отзыв: {review[:70]}...")
    print(f"  → {label} (вероятность: {proba:.4f})\n")

print("=" * 60)
print("ВСЁ ГОТОВО!")
print("=" * 60)
